# 03 — Data Cleaning

## Healthcare Claims & Revenue Analytics

This notebook transforms the validated and quality-assessed source data into
analysis-ready datasets.

### Objectives

- Preserve the original source data
- Standardize field data types
- Standardize date fields
- Preserve healthcare code representations
- Handle missing values according to documented rules
- Investigate financial-value anomalies
- Remove structurally unusable fields where appropriate
- Create analysis-ready datasets for SQL and Power BI

### Source Data

The source files are stored in:

`00_Working_Data/`

### Output Data

Cleaned datasets will be written to:

`02_Cleaned_Data/`

### Data Governance Principle

> **Raw source data will never be modified.**

All transformations will be performed on copies of the source data, and the
cleaned outputs will be saved separately.

### Cleaning Philosophy

Cleaning decisions will be based on the findings documented in:

`02_Data_Quality_Assessment.ipynb`

No value will be deleted or changed merely because it appears unusual.
Transformations must have a documented analytical or structural reason.

In [1]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Project Paths
# --------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics"
)

SOURCE_DIR = PROJECT_ROOT / "00_Working_Data"
CLEANED_DIR = PROJECT_ROOT / "02_Cleaned_Data"

# Validate source directory
if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SOURCE_DIR}"
    )

# Create cleaned-data directory if needed
CLEANED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print("\nSource directory:")
print(SOURCE_DIR)

print("\nCleaned-data directory:")
print(CLEANED_DIR)

print("\nSource directory exists:", SOURCE_DIR.exists())
print("Cleaned directory exists:", CLEANED_DIR.exists())

Project root:
C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics

Source directory:
C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics\00_Working_Data

Cleaned-data directory:
C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics\02_Cleaned_Data

Source directory exists: True
Cleaned directory exists: True


In [2]:
# --------------------------------------------------
# Source File Definitions
# --------------------------------------------------

files = {
    "beneficiary_2008": SOURCE_DIR / "DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv",
    "inpatient": SOURCE_DIR / "DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv",
    "outpatient": SOURCE_DIR / "DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv",
    "beneficiary_2009": SOURCE_DIR / "DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv",
    "beneficiary_2010": SOURCE_DIR / "DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv",
}

print("Source files:")

for name, file_path in files.items():
    print(f"{name:20} | {file_path.name} | Exists: {file_path.exists()}")

Source files:
beneficiary_2008     | DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv | Exists: True
inpatient            | DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv | Exists: True
outpatient           | DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv | Exists: True
beneficiary_2009     | DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv | Exists: True
beneficiary_2010     | DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv | Exists: True


## 1. Cleaning Rules

The following rules are based on the findings from
`02_Data_Quality_Assessment.ipynb`.

### Identifier Fields

- Preserve `DESYNPUF_ID` as a string.
- Preserve `CLM_ID` as a string.
- Preserve `SEGMENT` as a categorical/integer field.
- Do not remove records because of repeated `CLM_ID` values.
- The combination `CLM_ID + SEGMENT` will remain the claim-level business key.

### Date Fields

Convert applicable claim and beneficiary date fields to pandas datetime
format.

Invalid dates will be converted to missing values and reported rather than
silently replaced.

### Healthcare Codes

Store diagnosis, procedure, and HCPCS codes as strings.

This preserves:

- Leading zeros
- Alphanumeric codes
- Original code representations

Healthcare codes will not be treated as continuous numerical variables.

### Missing Values

Missing values will not automatically be replaced with zero.

The treatment of missing values will depend on the meaning and analytical
purpose of each field.

Fields that are structurally unavailable in the source extract will not be
artificially imputed.

### Financial Fields

Keep claim payment fields numeric.

- Retain negative payment values initially.
- Retain zero payment values.
- Do not automatically remove or replace unusual financial values.

Negative payment records will be flagged for further investigation.

### Inpatient Utilization

Keep `CLM_UTLZTN_DAY_CNT` numeric.

- Retain zero-day records.
- Retain missing values.
- No negative values were identified during quality assessment.

### Structurally Unavailable Fields

Completely unavailable fields may be excluded from the cleaned analytical
datasets where they provide no analytical value.

The raw source files will remain unchanged.

### Reproducibility

Every transformation applied to the source data will be explicitly defined
in this notebook and reflected in the resulting cleaned datasets.

In [3]:
# --------------------------------------------------
# Field Groups for Cleaning
# --------------------------------------------------

# Identifier fields
identifier_fields = [
    "DESYNPUF_ID",
    "CLM_ID",
    "PRVDR_NUM"
]

# Claim structure fields
claim_structure_fields = [
    "SEGMENT"
]

# Date fields
date_fields = [
    "CLM_FROM_DT",
    "CLM_THRU_DT",
    "BENE_BIRTH_DT",
    "BENE_DEATH_DT"
]

# Financial fields
financial_fields = [
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT"
]

# Inpatient utilization field
utilization_fields = [
    "CLM_UTLZTN_DAY_CNT"
]

# Healthcare code fields
code_prefixes = [
    "ICD9_DGNS_CD_",
    "ICD9_PRCDR_CD_",
    "HCPCS_CD_"
]

print("Identifier fields:")
print(identifier_fields)

print("\nClaim structure fields:")
print(claim_structure_fields)

print("\nDate fields:")
print(date_fields)

print("\nFinancial fields:")
print(financial_fields)

print("\nUtilization fields:")
print(utilization_fields)

print("\nHealthcare code prefixes:")
print(code_prefixes)

Identifier fields:
['DESYNPUF_ID', 'CLM_ID', 'PRVDR_NUM']

Claim structure fields:
['SEGMENT']

Date fields:
['CLM_FROM_DT', 'CLM_THRU_DT', 'BENE_BIRTH_DT', 'BENE_DEATH_DT']

Financial fields:
['CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT']

Utilization fields:
['CLM_UTLZTN_DAY_CNT']

Healthcare code prefixes:
['ICD9_DGNS_CD_', 'ICD9_PRCDR_CD_', 'HCPCS_CD_']


In [4]:
# --------------------------------------------------
# Reusable Claims Cleaning Function
# --------------------------------------------------

def clean_claims_chunk(chunk, dataset_name):
    """
    Apply standardized cleaning rules to one claims-data chunk.

    The original source data is not modified because the function operates
    on a DataFrame loaded from the source file.
    """

    # ----------------------------------------------
    # 1. Preserve identifiers as strings
    # ----------------------------------------------

    for column in identifier_fields:

        if column in chunk.columns:
            chunk[column] = (
                chunk[column]
                .astype("string")
                .str.strip()
            )

    # ----------------------------------------------
    # 2. Convert date fields
    # ----------------------------------------------

    for column in date_fields:

        if column in chunk.columns:

            chunk[column] = pd.to_datetime(
                chunk[column],
                errors="coerce"
            )

    # ----------------------------------------------
    # 3. Preserve healthcare codes as strings
    # ----------------------------------------------

    for column in chunk.columns:

        if any(
            column.startswith(prefix)
            for prefix in code_prefixes
        ):

            chunk[column] = (
                chunk[column]
                .astype("string")
                .str.strip()
            )

    # ----------------------------------------------
    # 4. Standardize SEGMENT
    # ----------------------------------------------

    if "SEGMENT" in chunk.columns:

        chunk["SEGMENT"] = pd.to_numeric(
            chunk["SEGMENT"],
            errors="coerce"
        ).astype("Int64")

    # ----------------------------------------------
    # 5. Standardize financial fields
    # ----------------------------------------------

    for column in financial_fields:

        if column in chunk.columns:

            chunk[column] = pd.to_numeric(
                chunk[column],
                errors="coerce"
            )

    # ----------------------------------------------
    # 6. Standardize utilization fields
    # ----------------------------------------------

    for column in utilization_fields:

        if column in chunk.columns:

            chunk[column] = pd.to_numeric(
                chunk[column],
                errors="coerce"
            )

    return chunk

In [5]:
# --------------------------------------------------
# Test Cleaning Function
# --------------------------------------------------

test_sample = pd.read_csv(
    files["inpatient"],
    nrows=10,
    low_memory=False
)

print("Before cleaning:")
print(test_sample.dtypes)

test_cleaned = clean_claims_chunk(
    test_sample.copy(),
    "inpatient"
)

print("\nAfter cleaning:")
print(test_cleaned.dtypes)

Before cleaning:
DESYNPUF_ID        str
CLM_ID           int64
SEGMENT          int64
CLM_FROM_DT      int64
CLM_THRU_DT      int64
                ...   
HCPCS_CD_41    float64
HCPCS_CD_42    float64
HCPCS_CD_43    float64
HCPCS_CD_44    float64
HCPCS_CD_45    float64
Length: 81, dtype: object

After cleaning:
DESYNPUF_ID            string
CLM_ID                 string
SEGMENT                 Int64
CLM_FROM_DT    datetime64[ns]
CLM_THRU_DT    datetime64[ns]
                    ...      
HCPCS_CD_41            string
HCPCS_CD_42            string
HCPCS_CD_43            string
HCPCS_CD_44            string
HCPCS_CD_45            string
Length: 81, dtype: object


In [6]:
# --------------------------------------------------
# Verify Cleaned Values
# --------------------------------------------------

verification_columns = [
    "DESYNPUF_ID",
    "CLM_ID",
    "SEGMENT",
    "CLM_FROM_DT",
    "CLM_THRU_DT",
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "ICD9_DGNS_CD_1",
    "ICD9_PRCDR_CD_1",
    "HCPCS_CD_1"
]

display(
    test_cleaned[verification_columns].head(10)
)

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,ICD9_DGNS_CD_1,ICD9_PRCDR_CD_1,HCPCS_CD_1
0,00013D2EFD8E45D1,196661176988405,1,1970-01-01 00:00:00.020100312,1970-01-01 00:00:00.020100313,4000.0,0.0,7802,<NA>,<NA>
1,00016F745862898F,196201177000368,1,1970-01-01 00:00:00.020090412,1970-01-01 00:00:00.020090418,26000.0,0.0,1970,<NA>,<NA>
2,00016F745862898F,196661177015632,1,1970-01-01 00:00:00.020090831,1970-01-01 00:00:00.020090902,5000.0,0.0,6186,7092.0,<NA>
3,00016F745862898F,196091176981058,1,1970-01-01 00:00:00.020090917,1970-01-01 00:00:00.020090920,5000.0,0.0,29623,<NA>,<NA>
4,00016F745862898F,196261176983265,1,1970-01-01 00:00:00.020100626,1970-01-01 00:00:00.020100701,16000.0,0.0,3569,<NA>,<NA>
5,00052705243EA128,196991176971757,1,1970-01-01 00:00:00.020080912,1970-01-01 00:00:00.020080912,14000.0,0.0,486,<NA>,<NA>
6,0007F12A492FD25D,196661176963773,1,1970-01-01 00:00:00.020080919,1970-01-01 00:00:00.020080922,5000.0,0.0,33811,8659.0,<NA>
7,0007F12A492FD25D,196821177025734,1,1970-01-01 00:00:00.020100602,1970-01-01 00:00:00.020100606,5000.0,0.0,49121,<NA>,<NA>
8,0007F12A492FD25D,196551177025145,1,1970-01-01 00:00:00.020100522,1970-01-01 00:00:00.020100612,14000.0,0.0,V5789,<NA>,<NA>
9,0007F12A492FD25D,196831176966961,1,1970-01-01 00:00:00.020100616,1970-01-01 00:00:00.020100619,29000.0,0.0,7366,73.0,<NA>


In [7]:
# --------------------------------------------------
# Corrected Claims Cleaning Function
# --------------------------------------------------

def clean_claims_chunk(chunk, dataset_name):
    """
    Apply standardized cleaning rules to one claims-data chunk.

    Identifiers, dates, and healthcare codes are initially read as strings
    so that leading zeros and alphanumeric values are preserved.
    """

    # ----------------------------------------------
    # 1. Clean identifier fields
    # ----------------------------------------------

    for column in identifier_fields:

        if column in chunk.columns:
            chunk[column] = (
                chunk[column]
                .astype("string")
                .str.strip()
            )

    # ----------------------------------------------
    # 2. Convert claim dates correctly
    # ----------------------------------------------

    for column in ["CLM_FROM_DT", "CLM_THRU_DT"]:

        if column in chunk.columns:

            chunk[column] = pd.to_datetime(
                chunk[column],
                format="%Y%m%d",
                errors="coerce"
            )

    # ----------------------------------------------
    # 3. Convert beneficiary dates correctly
    # ----------------------------------------------

    for column in ["BENE_BIRTH_DT", "BENE_DEATH_DT"]:

        if column in chunk.columns:

            chunk[column] = pd.to_datetime(
                chunk[column],
                format="%Y%m%d",
                errors="coerce"
            )

    # ----------------------------------------------
    # 4. Preserve healthcare codes as strings
    # ----------------------------------------------

    for column in chunk.columns:

        if any(
            column.startswith(prefix)
            for prefix in code_prefixes
        ):

            chunk[column] = (
                chunk[column]
                .astype("string")
                .str.strip()
            )

    # ----------------------------------------------
    # 5. Standardize SEGMENT
    # ----------------------------------------------

    if "SEGMENT" in chunk.columns:

        chunk["SEGMENT"] = pd.to_numeric(
            chunk["SEGMENT"],
            errors="coerce"
        ).astype("Int64")

    # ----------------------------------------------
    # 6. Standardize financial fields
    # ----------------------------------------------

    for column in financial_fields:

        if column in chunk.columns:

            chunk[column] = pd.to_numeric(
                chunk[column],
                errors="coerce"
            )

    # ----------------------------------------------
    # 7. Standardize utilization fields
    # ----------------------------------------------

    for column in utilization_fields:

        if column in chunk.columns:

            chunk[column] = pd.to_numeric(
                chunk[column],
                errors="coerce"
            )

    return chunk

In [8]:
# --------------------------------------------------
# Test Source Loading with String Preservation
# --------------------------------------------------

test_sample = pd.read_csv(
    files["inpatient"],
    nrows=10,
    dtype=str,
    low_memory=False
)

print("Source values before cleaning:")

display(
    test_sample[
        [
            "CLM_FROM_DT",
            "CLM_THRU_DT",
            "ICD9_PRCDR_CD_1"
        ]
    ]
)

test_cleaned = clean_claims_chunk(
    test_sample.copy(),
    "inpatient"
)

print("\nCleaned values:")

display(
    test_cleaned[
        [
            "CLM_FROM_DT",
            "CLM_THRU_DT",
            "ICD9_PRCDR_CD_1"
        ]
    ]
)

Source values before cleaning:


,CLM_FROM_DT,CLM_THRU_DT,ICD9_PRCDR_CD_1
0,20100312,20100313,NaN
1,20090412,20090418,NaN
2,20090831,20090902,7092
3,20090917,20090920,NaN
4,20100626,20100701,NaN
5,20080912,20080912,NaN
6,20080919,20080922,8659
7,20100602,20100606,NaN
8,20100522,20100612,NaN
9,20100616,20100619,0073



Cleaned values:


,CLM_FROM_DT,CLM_THRU_DT,ICD9_PRCDR_CD_1
0,2010-03-12,2010-03-13,<NA>
1,2009-04-12,2009-04-18,<NA>
2,2009-08-31,2009-09-02,7092
3,2009-09-17,2009-09-20,<NA>
4,2010-06-26,2010-07-01,<NA>
5,2008-09-12,2008-09-12,<NA>
6,2008-09-19,2008-09-22,8659
7,2010-06-02,2010-06-06,<NA>
8,2010-05-22,2010-06-12,<NA>
9,2010-06-16,2010-06-19,0073


In [9]:
# --------------------------------------------------
# Verify Identifier, Financial, and Utilization Fields
# --------------------------------------------------

display(
    test_cleaned[
        [
            "DESYNPUF_ID",
            "CLM_ID",
            "SEGMENT",
            "CLM_PMT_AMT",
            "NCH_PRMRY_PYR_CLM_PD_AMT",
            "CLM_UTLZTN_DAY_CNT"
        ]
    ]
)

print("\nData types after cleaning:")

print(
    test_cleaned[
        [
            "DESYNPUF_ID",
            "CLM_ID",
            "SEGMENT",
            "CLM_PMT_AMT",
            "NCH_PRMRY_PYR_CLM_PD_AMT",
            "CLM_UTLZTN_DAY_CNT"
        ]
    ].dtypes
)

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,CLM_UTLZTN_DAY_CNT
0,00013D2EFD8E45D1,196661176988405,1,4000.0,0.0,1
1,00016F745862898F,196201177000368,1,26000.0,0.0,6
2,00016F745862898F,196661177015632,1,5000.0,0.0,2
3,00016F745862898F,196091176981058,1,5000.0,0.0,3
4,00016F745862898F,196261176983265,1,16000.0,0.0,5
5,00052705243EA128,196991176971757,1,14000.0,0.0,1
6,0007F12A492FD25D,196661176963773,1,5000.0,0.0,3
7,0007F12A492FD25D,196821177025734,1,5000.0,0.0,4
8,0007F12A492FD25D,196551177025145,1,14000.0,0.0,21
9,0007F12A492FD25D,196831176966961,1,29000.0,0.0,3



Data types after cleaning:
DESYNPUF_ID                  string
CLM_ID                       string
SEGMENT                       Int64
CLM_PMT_AMT                 float64
NCH_PRMRY_PYR_CLM_PD_AMT    float64
CLM_UTLZTN_DAY_CNT            int64
dtype: object


In [10]:
# --------------------------------------------------
# Test Cleaning on Outpatient Claims
# --------------------------------------------------

outpatient_sample = pd.read_csv(
    files["outpatient"],
    nrows=10,
    dtype=str,
    low_memory=False
)

outpatient_cleaned = clean_claims_chunk(
    outpatient_sample.copy(),
    "outpatient"
)

print("Outpatient healthcare-code sample:")

display(
    outpatient_cleaned[
        [
            "HCPCS_CD_1",
            "HCPCS_CD_2",
            "HCPCS_CD_3",
            "ICD9_DGNS_CD_1"
        ]
    ]
)

print("\nData types:")

print(
    outpatient_cleaned[
        [
            "DESYNPUF_ID",
            "CLM_ID",
            "SEGMENT",
            "CLM_FROM_DT",
            "CLM_PMT_AMT",
            "HCPCS_CD_1",
            "ICD9_DGNS_CD_1"
        ]
    ].dtypes
)

Outpatient healthcare-code sample:


,HCPCS_CD_1,HCPCS_CD_2,HCPCS_CD_3,ICD9_DGNS_CD_1
0,85610,84153,<NA>,V5841
1,85610,80048,80061,V5832
2,71101,78480,94060,9594
3,36415,<NA>,<NA>,78943
4,76872,82365,<NA>,6009
5,G0206,<NA>,<NA>,6115
6,36415,80076,<NA>,2723
7,G0202,77052,<NA>,V821
8,73562,<NA>,<NA>,<NA>
9,36415,85007,<NA>,7902



Data types:
DESYNPUF_ID               string
CLM_ID                    string
SEGMENT                    Int64
CLM_FROM_DT       datetime64[us]
CLM_PMT_AMT              float64
HCPCS_CD_1                string
ICD9_DGNS_CD_1            string
dtype: object


## 2. Full Dataset Processing

The validated cleaning function will now be applied to the complete source
datasets.

Large claims files will be processed in chunks to avoid loading the entire
dataset into memory at once.

### Processing Principles

- Source files will be read from `00_Working_Data/`.
- Source files will not be modified.
- All source fields will initially be read as strings.
- The standardized cleaning function will be applied to each chunk.
- Cleaned outputs will be written to `02_Cleaned_Data/`.
- Existing cleaned output files will be replaced only when this notebook is
  intentionally rerun.
- Record counts will be tracked during processing.
- The final cleaned datasets will be validated against the source record
  counts.

In [11]:
# --------------------------------------------------
# Chunk-Based Claims Processing Function
# --------------------------------------------------

def process_claims_file(
    source_path,
    output_path,
    dataset_name,
    chunk_size=50000
):
    """
    Process a large claims CSV file in chunks.

    The source file is never modified.
    Cleaned data is written to a separate output CSV.
    """

    total_rows = 0
    first_chunk = True

    for chunk in pd.read_csv(
        source_path,
        dtype=str,
        chunksize=chunk_size,
        low_memory=False
    ):

        # Apply validated cleaning function
        cleaned_chunk = clean_claims_chunk(
            chunk.copy(),
            dataset_name
        )

        # Write first chunk with column headers
        cleaned_chunk.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        total_rows += len(cleaned_chunk)
        first_chunk = False

        print(
            f"{dataset_name}: "
            f"{total_rows:,} rows processed"
        )

    print("\nProcessing complete.")
    print(f"Dataset: {dataset_name}")
    print(f"Rows processed: {total_rows:,}")
    print(f"Output: {output_path}")

    return total_rows

In [12]:
# --------------------------------------------------
# Process Inpatient Claims
# --------------------------------------------------

inpatient_output = (
    CLEANED_DIR / "Inpatient_Claims_Cleaned.csv"
)

inpatient_rows = process_claims_file(
    source_path=files["inpatient"],
    output_path=inpatient_output,
    dataset_name="inpatient"
)

inpatient: 50,000 rows processed
inpatient: 66,773 rows processed

Processing complete.
Dataset: inpatient
Rows processed: 66,773
Output: C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics\02_Cleaned_Data\Inpatient_Claims_Cleaned.csv


In [13]:
# --------------------------------------------------
# Verify Cleaned Inpatient Claims
# --------------------------------------------------

cleaned_inpatient = pd.read_csv(
    inpatient_output,
    dtype=str,
    low_memory=False
)

print("Cleaned inpatient rows:", f"{len(cleaned_inpatient):,}")
print("Cleaned inpatient columns:", len(cleaned_inpatient.columns))

print("\nExpected source rows: 66,773")
print("Row count matches:", len(cleaned_inpatient) == 66773)

print("\nSelected cleaned values:")

display(
    cleaned_inpatient[
        [
            "DESYNPUF_ID",
            "CLM_ID",
            "SEGMENT",
            "CLM_FROM_DT",
            "CLM_THRU_DT",
            "CLM_PMT_AMT",
            "CLM_UTLZTN_DAY_CNT",
            "ICD9_PRCDR_CD_1"
        ]
    ].head(10)
)

Cleaned inpatient rows: 66,773
Cleaned inpatient columns: 81

Expected source rows: 66,773
Row count matches: True

Selected cleaned values:


,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,CLM_PMT_AMT,CLM_UTLZTN_DAY_CNT,ICD9_PRCDR_CD_1
0,00013D2EFD8E45D1,196661176988405,1,2010-03-12,2010-03-13,4000.0,1.0,NaN
1,00016F745862898F,196201177000368,1,2009-04-12,2009-04-18,26000.0,6.0,NaN
2,00016F745862898F,196661177015632,1,2009-08-31,2009-09-02,5000.0,2.0,7092
3,00016F745862898F,196091176981058,1,2009-09-17,2009-09-20,5000.0,3.0,NaN
4,00016F745862898F,196261176983265,1,2010-06-26,2010-07-01,16000.0,5.0,NaN
5,00052705243EA128,196991176971757,1,2008-09-12,2008-09-12,14000.0,1.0,NaN
6,0007F12A492FD25D,196661176963773,1,2008-09-19,2008-09-22,5000.0,3.0,8659
7,0007F12A492FD25D,196821177025734,1,2010-06-02,2010-06-06,5000.0,4.0,NaN
8,0007F12A492FD25D,196551177025145,1,2010-05-22,2010-06-12,14000.0,21.0,NaN
9,0007F12A492FD25D,196831176966961,1,2010-06-16,2010-06-19,29000.0,3.0,0073


In [14]:
# --------------------------------------------------
# Process Outpatient Claims
# --------------------------------------------------

outpatient_output = (
    CLEANED_DIR / "Outpatient_Claims_Cleaned.csv"
)

outpatient_rows = process_claims_file(
    source_path=files["outpatient"],
    output_path=outpatient_output,
    dataset_name="outpatient"
)

outpatient: 50,000 rows processed
outpatient: 100,000 rows processed
outpatient: 150,000 rows processed
outpatient: 200,000 rows processed
outpatient: 250,000 rows processed
outpatient: 300,000 rows processed
outpatient: 350,000 rows processed
outpatient: 400,000 rows processed
outpatient: 450,000 rows processed
outpatient: 500,000 rows processed
outpatient: 550,000 rows processed
outpatient: 600,000 rows processed
outpatient: 650,000 rows processed
outpatient: 700,000 rows processed
outpatient: 750,000 rows processed
outpatient: 790,790 rows processed

Processing complete.
Dataset: outpatient
Rows processed: 790,790
Output: C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics\02_Cleaned_Data\Outpatient_Claims_Cleaned.csv


In [ ]:
# --------------------------------------------------
# Verify Cleaned Outpatient Claims
# --------------------------------------------------

cleaned_outpatient = pd.read_csv(
    outpatient_output,
    dtype=str,
    low_memory=False
)

print("Cleaned outpatient rows:", f"{len(cleaned_outpatient):,}")
print("Cleaned outpatient columns:", len(cleaned_outpatient.columns))

print("\nExpected source rows: 790,790")
print("Row count matches:", len(cleaned_outpatient) == 790790)

print("\nSelected cleaned values:")

display(
    cleaned_outpatient[
        [
            "DESYNPUF_ID",
            "CLM_ID",
            "SEGMENT",
            "CLM_FROM_DT",
            "CLM_THRU_DT",
            "CLM_PMT_AMT",
            "HCPCS_CD_1",
            "HCPCS_CD_2",
            "ICD9_DGNS_CD_1"
        ]
    ].head(10)
)

In [ ]:
# --------------------------------------------------
# Beneficiary Cleaning Function
# --------------------------------------------------

def clean_beneficiary_chunk(chunk, dataset_name):
    """
    Apply standardized cleaning rules to one beneficiary-data chunk.

    Identifiers and coded healthcare fields are preserved as strings.
    Date fields are converted from YYYYMMDD to pandas datetime.
    """

    # 1. Clean beneficiary identifier
    if "DESYNPUF_ID" in chunk.columns:
        chunk["DESYNPUF_ID"] = (
            chunk["DESYNPUF_ID"]
            .astype("string")
            .str.strip()
        )

    # 2. Convert beneficiary dates correctly
    for column in ["BENE_BIRTH_DT", "BENE_DEATH_DT"]:
        if column in chunk.columns:
            chunk[column] = pd.to_datetime(
                chunk[column],
                format="%Y%m%d",
                errors="coerce"
            )

    # 3. Preserve coded fields as strings
    beneficiary_code_prefixes = [
        "BENE_",
        "SP_",
        "IP_",
        "OP_"
    ]

    for column in chunk.columns:
        if any(
            column.startswith(prefix)
            for prefix in beneficiary_code_prefixes
        ):
            if column not in [
                "BENE_BIRTH_DT",
                "BENE_DEATH_DT"
            ]:
                chunk[column] = (
                    chunk[column]
                    .astype("string")
                    .str.strip()
                )

    return chunk

In [ ]:
# --------------------------------------------------
# Test Beneficiary Cleaning
# --------------------------------------------------

beneficiary_sample = pd.read_csv(
    files["beneficiary_2008"],
    nrows=10,
    dtype=str,
    low_memory=False
)

print("Source beneficiary values:")

display(
    beneficiary_sample[
        [
            "DESYNPUF_ID",
            "BENE_BIRTH_DT",
            "BENE_DEATH_DT",
            "BENE_SEX_IDENT_CD",
            "BENE_RACE_CD",
            "BENE_ESRD_IND"
        ]
    ]
)

beneficiary_cleaned = clean_beneficiary_chunk(
    beneficiary_sample.copy(),
    "beneficiary_2008"
)

print("\nCleaned beneficiary values:")

display(
    beneficiary_cleaned[
        [
            "DESYNPUF_ID",
            "BENE_BIRTH_DT",
            "BENE_DEATH_DT",
            "BENE_SEX_IDENT_CD",
            "BENE_RACE_CD",
            "BENE_ESRD_IND"
        ]
    ]
)

Source beneficiary values:


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND
0,00013D2EFD8E45D1,19230501,NaN,1,1,0
1,00016F745862898F,19430101,NaN,1,1,0
2,0001FDD721E223DC,19360901,NaN,2,1,0
3,00021CA6FF03E670,19410601,NaN,1,5,0
4,00024B3D2352D2D0,19360801,NaN,1,1,0
5,0002DAE1C81CC70D,19431001,NaN,1,2,0
6,0002F28CE057345B,19220701,NaN,1,1,0
7,000308435E3E5B76,19350901,NaN,1,1,0
8,000345A39D4157C9,19760901,NaN,2,1,0
9,00036A21B65B0206,19381001,NaN,2,2,0



Cleaned beneficiary values:


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND
0,00013D2EFD8E45D1,1923-05-01,NaT,1,1,0
1,00016F745862898F,1943-01-01,NaT,1,1,0
2,0001FDD721E223DC,1936-09-01,NaT,2,1,0
3,00021CA6FF03E670,1941-06-01,NaT,1,5,0
4,00024B3D2352D2D0,1936-08-01,NaT,1,1,0
5,0002DAE1C81CC70D,1943-10-01,NaT,1,2,0
6,0002F28CE057345B,1922-07-01,NaT,1,1,0
7,000308435E3E5B76,1935-09-01,NaT,1,1,0
8,000345A39D4157C9,1976-09-01,NaT,2,1,0
9,00036A21B65B0206,1938-10-01,NaT,2,2,0


In [ ]:
# --------------------------------------------------
# Chunk-Based Beneficiary Processing Function
# --------------------------------------------------

def process_beneficiary_file(
    source_path,
    output_path,
    dataset_name,
    chunk_size=50000
):
    """
    Process a beneficiary CSV file in chunks.

    The source file is never modified.
    Cleaned data is written to a separate output CSV.
    """

    total_rows = 0
    first_chunk = True

    for chunk in pd.read_csv(
        source_path,
        dtype=str,
        chunksize=chunk_size,
        low_memory=False
    ):

        # Apply beneficiary cleaning function
        cleaned_chunk = clean_beneficiary_chunk(
            chunk.copy(),
            dataset_name
        )

        # Write cleaned chunk
        cleaned_chunk.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        total_rows += len(cleaned_chunk)
        first_chunk = False

        print(
            f"{dataset_name}: "
            f"{total_rows:,} rows processed"
        )

    print("\nProcessing complete.")
    print(f"Dataset: {dataset_name}")
    print(f"Rows processed: {total_rows:,}")
    print(f"Output: {output_path}")

    return total_rows

In [ ]:
# --------------------------------------------------
# Process 2008 Beneficiary Data
# --------------------------------------------------

beneficiary_2008_output = (
    CLEANED_DIR / "Beneficiary_2008_Cleaned.csv"
)

beneficiary_2008_rows = process_beneficiary_file(
    source_path=files["beneficiary_2008"],
    output_path=beneficiary_2008_output,
    dataset_name="beneficiary_2008"
)

beneficiary_2008: 50,000 rows processed
beneficiary_2008: 100,000 rows processed
beneficiary_2008: 116,352 rows processed

Processing complete.
Dataset: beneficiary_2008
Rows processed: 116,352
Output: C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics\02_Cleaned_Data\Beneficiary_2008_Cleaned.csv


In [ ]:
# --------------------------------------------------
# Verify Cleaned 2008 Beneficiary Data
# --------------------------------------------------

cleaned_beneficiary_2008 = pd.read_csv(
    beneficiary_2008_output,
    dtype=str,
    low_memory=False
)

print(
    "Cleaned 2008 beneficiary rows:",
    f"{len(cleaned_beneficiary_2008):,}"
)

print(
    "Cleaned 2008 beneficiary columns:",
    len(cleaned_beneficiary_2008.columns)
)

print("\nExpected source rows: 116,352")
print(
    "Row count matches:",
    len(cleaned_beneficiary_2008) == 116352
)

print("\nSelected cleaned values:")

display(
    cleaned_beneficiary_2008[
        [
            "DESYNPUF_ID",
            "BENE_BIRTH_DT",
            "BENE_DEATH_DT",
            "BENE_SEX_IDENT_CD",
            "BENE_RACE_CD",
            "BENE_ESRD_IND"
        ]
    ].head(10)
)

Cleaned 2008 beneficiary rows: 116,352
Cleaned 2008 beneficiary columns: 32

Expected source rows: 116,352
Row count matches: True

Selected cleaned values:


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND
0,00013D2EFD8E45D1,1923-05-01,NaN,1,1,0
1,00016F745862898F,1943-01-01,NaN,1,1,0
2,0001FDD721E223DC,1936-09-01,NaN,2,1,0
3,00021CA6FF03E670,1941-06-01,NaN,1,5,0
4,00024B3D2352D2D0,1936-08-01,NaN,1,1,0
5,0002DAE1C81CC70D,1943-10-01,NaN,1,2,0
6,0002F28CE057345B,1922-07-01,NaN,1,1,0
7,000308435E3E5B76,1935-09-01,NaN,1,1,0
8,000345A39D4157C9,1976-09-01,NaN,2,1,0
9,00036A21B65B0206,1938-10-01,NaN,2,2,0


In [ ]:
# --------------------------------------------------
# Process 2009 and 2010 Beneficiary Data
# --------------------------------------------------

beneficiary_2009_output = (
    CLEANED_DIR / "Beneficiary_2009_Cleaned.csv"
)

beneficiary_2010_output = (
    CLEANED_DIR / "Beneficiary_2010_Cleaned.csv"
)

beneficiary_2009_rows = process_beneficiary_file(
    source_path=files["beneficiary_2009"],
    output_path=beneficiary_2009_output,
    dataset_name="beneficiary_2009"
)

print("\n" + "=" * 60 + "\n")

beneficiary_2010_rows = process_beneficiary_file(
    source_path=files["beneficiary_2010"],
    output_path=beneficiary_2010_output,
    dataset_name="beneficiary_2010"
)

beneficiary_2009: 50,000 rows processed
beneficiary_2009: 100,000 rows processed
beneficiary_2009: 114,538 rows processed

Processing complete.
Dataset: beneficiary_2009
Rows processed: 114,538
Output: C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics\02_Cleaned_Data\Beneficiary_2009_Cleaned.csv


beneficiary_2010: 50,000 rows processed
beneficiary_2010: 100,000 rows processed
beneficiary_2010: 112,754 rows processed

Processing complete.
Dataset: beneficiary_2010
Rows processed: 112,754
Output: C:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics\02_Cleaned_Data\Beneficiary_2010_Cleaned.csv


In [ ]:
# --------------------------------------------------
# Verify 2009 and 2010 Beneficiary Outputs
# --------------------------------------------------

cleaned_beneficiary_2009 = pd.read_csv(
    beneficiary_2009_output,
    dtype=str,
    low_memory=False
)

cleaned_beneficiary_2010 = pd.read_csv(
    beneficiary_2010_output,
    dtype=str,
    low_memory=False
)

verification = pd.DataFrame({
    "Dataset": [
        "Beneficiary 2008",
        "Beneficiary 2009",
        "Beneficiary 2010"
    ],
    "Expected Rows": [
        116352,
        114538,
        112754
    ],
    "Actual Rows": [
        len(cleaned_beneficiary_2008),
        len(cleaned_beneficiary_2009),
        len(cleaned_beneficiary_2010)
    ],
    "Columns": [
        len(cleaned_beneficiary_2008.columns),
        len(cleaned_beneficiary_2009.columns),
        len(cleaned_beneficiary_2010.columns)
    ]
})

verification["Row Count Matches"] = (
    verification["Expected Rows"]
    == verification["Actual Rows"]
)

display(verification)

,Dataset,Expected Rows,Actual Rows,Columns,Row Count Matches
0,Beneficiary 2008,116352,116352,32,True
1,Beneficiary 2009,114538,114538,32,True
2,Beneficiary 2010,112754,112754,32,True


## 3. Post-Cleaning Validation

After cleaning, the resulting datasets are revalidated against the source
data-quality findings.

The purpose of this step is to confirm that cleaning transformations:

- did not remove records unintentionally;
- did not introduce duplicate business keys;
- preserved beneficiary and claim identifiers;
- preserved healthcare code representations;
- produced valid date fields;
- retained financial values without inappropriate modification;
- maintained the expected relationships between beneficiaries and claims.

A successful post-cleaning validation confirms that the datasets are ready
for exploratory analysis, SQL analysis, and Power BI development.

In [ ]:
# --------------------------------------------------
# Post-Cleaning Validation: Claim Business Keys
# --------------------------------------------------

inpatient_duplicate_keys = (
    cleaned_inpatient
    .duplicated(subset=["CLM_ID", "SEGMENT"])
    .sum()
)

outpatient_duplicate_keys = (
    cleaned_outpatient
    .duplicated(subset=["CLM_ID", "SEGMENT"])
    .sum()
)

claim_key_validation = pd.DataFrame({
    "Dataset": [
        "Inpatient Claims",
        "Outpatient Claims"
    ],
    "Rows": [
        len(cleaned_inpatient),
        len(cleaned_outpatient)
    ],
    "Duplicate CLM_ID + SEGMENT Keys": [
        inpatient_duplicate_keys,
        outpatient_duplicate_keys
    ],
    "Business Key Unique": [
        inpatient_duplicate_keys == 0,
        outpatient_duplicate_keys == 0
    ]
})

display(claim_key_validation)

,Dataset,Rows,Duplicate CLM_ID + SEGMENT Keys,Business Key Unique
0,Inpatient Claims,66773,0,True
1,Outpatient Claims,790790,0,True


In [ ]:
# --------------------------------------------------
# Post-Cleaning Validation: Beneficiary Identifiers
# --------------------------------------------------

beneficiary_validation = []

for dataset_name, dataframe in [
    ("Beneficiary 2008", cleaned_beneficiary_2008),
    ("Beneficiary 2009", cleaned_beneficiary_2009),
    ("Beneficiary 2010", cleaned_beneficiary_2010)
]:

    missing_ids = dataframe["DESYNPUF_ID"].isna().sum()

    duplicate_ids = dataframe["DESYNPUF_ID"].duplicated().sum()

    beneficiary_validation.append({
        "Dataset": dataset_name,
        "Rows": len(dataframe),
        "Missing DESYNPUF_ID": missing_ids,
        "Duplicate DESYNPUF_ID": duplicate_ids,
        "Identifier Valid": (
            missing_ids == 0
            and duplicate_ids == 0
        )
    })

beneficiary_validation = pd.DataFrame(
    beneficiary_validation
)

display(beneficiary_validation)

,Dataset,Rows,Missing DESYNPUF_ID,Duplicate DESYNPUF_ID,Identifier Valid
0,Beneficiary 2008,116352,0,0,True
1,Beneficiary 2009,114538,0,0,True
2,Beneficiary 2010,112754,0,0,True


In [ ]:
# --------------------------------------------------
# Post-Cleaning Validation: Claim Dates
# --------------------------------------------------

date_validation = []

for dataset_name, dataframe in [
    ("Inpatient Claims", cleaned_inpatient),
    ("Outpatient Claims", cleaned_outpatient)
]:

    from_dates = pd.to_datetime(
        dataframe["CLM_FROM_DT"],
        errors="coerce"
    )

    thru_dates = pd.to_datetime(
        dataframe["CLM_THRU_DT"],
        errors="coerce"
    )

    missing_from = from_dates.isna().sum()
    missing_thru = thru_dates.isna().sum()

    inverted_dates = (
        (thru_dates < from_dates)
        & from_dates.notna()
        & thru_dates.notna()
    ).sum()

    date_validation.append({
        "Dataset": dataset_name,
        "Missing CLM_FROM_DT": missing_from,
        "Missing CLM_THRU_DT": missing_thru,
        "Inverted Date Records": inverted_dates,
        "Date Validation Passed": inverted_dates == 0
    })

date_validation = pd.DataFrame(date_validation)

display(date_validation)

,Dataset,Missing CLM_FROM_DT,Missing CLM_THRU_DT,Inverted Date Records,Date Validation Passed
0,Inpatient Claims,68,68,0,True
1,Outpatient Claims,11253,11253,0,True


In [ ]:
# --------------------------------------------------
# Post-Cleaning Validation: Financial Fields
# --------------------------------------------------

financial_validation = []

for dataset_name, dataframe in [
    ("Inpatient Claims", cleaned_inpatient),
    ("Outpatient Claims", cleaned_outpatient)
]:

    payment = pd.to_numeric(
        dataframe["CLM_PMT_AMT"],
        errors="coerce"
    )

    primary_payer = pd.to_numeric(
        dataframe["NCH_PRMRY_PYR_CLM_PD_AMT"],
        errors="coerce"
    )

    financial_validation.append({
        "Dataset": dataset_name,
        "Missing Claim Payments": payment.isna().sum(),
        "Negative Claim Payments": (payment < 0).sum(),
        "Zero Claim Payments": (payment == 0).sum(),
        "Minimum Claim Payment": payment.min(),
        "Maximum Claim Payment": payment.max(),
        "Total Claim Payments": payment.sum(),
        "Missing Primary Payer Payments": primary_payer.isna().sum(),
        "Negative Primary Payer Payments": (primary_payer < 0).sum(),
        "Zero Primary Payer Payments": (primary_payer == 0).sum()
    })

financial_validation = pd.DataFrame(
    financial_validation
)

display(financial_validation)

,Dataset,Missing Claim Payments,Negative Claim Payments,Zero Claim Payments,Minimum Claim Payment,Maximum Claim Payment,Total Claim Payments,Missing Primary Payer Payments,Negative Primary Payer Payments,Zero Primary Payer Payments
0,Inpatient Claims,0,55,2160,-8000.0,57000.0,639260180.0,0,0,65246
1,Outpatient Claims,0,2566,30105,-100.0,3300.0,224524710.0,0,0,783826


In [ ]:
# --------------------------------------------------
# Post-Cleaning Validation: Healthcare Codes
# --------------------------------------------------

code_validation = []

# Representative code fields
code_fields = {
    "Inpatient Claims": [
        "ICD9_DGNS_CD_1",
        "ICD9_PRCDR_CD_1"
    ],
    "Outpatient Claims": [
        "ICD9_DGNS_CD_1",
        "HCPCS_CD_1"
    ]
}

datasets = {
    "Inpatient Claims": cleaned_inpatient,
    "Outpatient Claims": cleaned_outpatient
}

for dataset_name, columns in code_fields.items():

    dataframe = datasets[dataset_name]

    for column in columns:

        non_missing = dataframe[column].dropna()

        numeric_like = pd.to_numeric(
            non_missing,
            errors="coerce"
        ).notna().sum()

        code_validation.append({
            "Dataset": dataset_name,
            "Field": column,
            "Non-Missing Values": len(non_missing),
            "String Values": (
                non_missing.map(
                    lambda x: isinstance(x, str)
                ).sum()
            ),
            "Numeric-Looking Values": numeric_like
        })

code_validation = pd.DataFrame(code_validation)

display(code_validation)

,Dataset,Field,Non-Missing Values,String Values,Numeric-Looking Values
0,Inpatient Claims,ICD9_DGNS_CD_1,66678,66678,64521
1,Inpatient Claims,ICD9_PRCDR_CD_1,38231,38231,38231
2,Outpatient Claims,ICD9_DGNS_CD_1,785170,785170,620076
3,Outpatient Claims,HCPCS_CD_1,646260,646260,567728


In [ ]:
# --------------------------------------------------
# Post-Cleaning Validation: Beneficiary-Claims Relationship
# --------------------------------------------------

# Create master beneficiary ID set across all annual files
beneficiary_ids = set(
    pd.concat(
        [
            cleaned_beneficiary_2008["DESYNPUF_ID"],
            cleaned_beneficiary_2009["DESYNPUF_ID"],
            cleaned_beneficiary_2010["DESYNPUF_ID"]
        ],
        ignore_index=True
    )
    .dropna()
    .unique()
)

relationship_validation = []

for dataset_name, dataframe in [
    ("Inpatient Claims", cleaned_inpatient),
    ("Outpatient Claims", cleaned_outpatient)
]:

    claim_beneficiary_ids = set(
        dataframe["DESYNPUF_ID"]
        .dropna()
        .unique()
    )

    unmatched_ids = claim_beneficiary_ids - beneficiary_ids

    relationship_validation.append({
        "Dataset": dataset_name,
        "Unique Beneficiary IDs in Claims": len(
            claim_beneficiary_ids
        ),
        "Unmatched Beneficiary IDs": len(
            unmatched_ids
        ),
        "Relationship Valid": len(unmatched_ids) == 0
    })

relationship_validation = pd.DataFrame(
    relationship_validation
)

print(
    "Master unique beneficiary IDs:",
    f"{len(beneficiary_ids):,}"
)

display(relationship_validation)

Master unique beneficiary IDs: 116,352


,Dataset,Unique Beneficiary IDs in Claims,Unmatched Beneficiary IDs,Relationship Valid
0,Inpatient Claims,37780,0,True
1,Outpatient Claims,85272,0,True


In [ ]:
# --------------------------------------------------
# Post-Cleaning Validation: Inpatient Utilization
# --------------------------------------------------

utilization = pd.to_numeric(
    cleaned_inpatient["CLM_UTLZTN_DAY_CNT"],
    errors="coerce"
)

utilization_validation = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Missing Utilization",
        "Negative Utilization",
        "Zero-Day Records",
        "Positive-Day Records",
        "Minimum Utilization",
        "Maximum Utilization"
    ],
    "Value": [
        len(utilization),
        utilization.isna().sum(),
        (utilization < 0).sum(),
        (utilization == 0).sum(),
        (utilization > 0).sum(),
        utilization.min(),
        utilization.max()
    ]
})

display(utilization_validation)

,Metric,Value
0,Total Records,66773.0
1,Missing Utilization,68.0
2,Negative Utilization,0.0
3,Zero-Day Records,2266.0
4,Positive-Day Records,64439.0
5,Minimum Utilization,0.0
6,Maximum Utilization,136.0


In [ ]:
# --------------------------------------------------
# Post-Cleaning Validation: Cleaned File Inventory
# --------------------------------------------------

expected_cleaned_files = {
    "Beneficiary 2008": beneficiary_2008_output,
    "Beneficiary 2009": beneficiary_2009_output,
    "Beneficiary 2010": beneficiary_2010_output,
    "Inpatient Claims": inpatient_output,
    "Outpatient Claims": outpatient_output
}

file_inventory = []

for dataset_name, file_path in expected_cleaned_files.items():

    file_inventory.append({
        "Dataset": dataset_name,
        "File": file_path.name,
        "Exists": file_path.exists(),
        "Size (MB)": round(
            file_path.stat().st_size / (1024 ** 2), 2
        ) if file_path.exists() else 0,
        "File Valid": (
            file_path.exists()
            and file_path.stat().st_size > 0
        )
    })

file_inventory = pd.DataFrame(file_inventory)

display(file_inventory)

,Dataset,File,Exists,Size (MB),File Valid
0,Beneficiary 2008,Beneficiary_2008_Cleaned.csv,True,14.14,True
1,Beneficiary 2009,Beneficiary_2009_Cleaned.csv,True,14.04,True
2,Beneficiary 2010,Beneficiary_2010_Cleaned.csv,True,13.65,True
3,Inpatient Claims,Inpatient_Claims_Cleaned.csv,True,16.17,True
4,Outpatient Claims,Outpatient_Claims_Cleaned.csv,True,155.78,True


## 4. Post-Cleaning Validation Summary

The cleaned datasets successfully passed the post-cleaning validation checks.

### Structural Validation

- Inpatient claims: 66,773 records retained.
- Outpatient claims: 790,790 records retained.
- `CLM_ID + SEGMENT` remains unique in both claims datasets.
- Beneficiary files retained their expected annual record counts.
- `DESYNPUF_ID` contains no missing or duplicate values within each annual
  beneficiary file.

### Relationship Validation

- 116,352 unique beneficiary IDs were identified across the three annual
  beneficiary files.
- All beneficiary IDs referenced by inpatient claims were found in the
  beneficiary master set.
- All beneficiary IDs referenced by outpatient claims were found in the
  beneficiary master set.

### Date Validation

- Claim dates were successfully standardized.
- No inverted claim periods were identified.
- Source-level missing claim dates were preserved.

### Financial Validation

- No claim payment values are missing.
- Negative payment values were retained for investigation.
- Zero payment values were retained.
- Financial totals and ranges remain consistent with the earlier
  Data Quality Assessment.

### Healthcare Code Validation

- Healthcare diagnosis, procedure, and HCPCS fields remain represented as
  strings.
- Leading zeros and alphanumeric healthcare codes were preserved.
- Missing healthcare codes were not artificially imputed.

### Inpatient Utilization Validation

- No negative utilization values were identified.
- Zero-day and positive-day records were retained.
- Missing utilization values were preserved.

### Output Validation

All five cleaned datasets were successfully written to:

`02_Cleaned_Data/`

The cleaned datasets are therefore ready for the next analytical stage.

> **Important:** Negative payment values and missing values have been retained
> intentionally. They will be handled according to their analytical meaning
> during exploratory analysis rather than removed automatically.